In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import import_before_profile
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tensorflow_probability.substrates import jax as tfp

from qdots_qll.distributions import (
    Distribution,
)
from qdots_qll.exp_design import RandExpDesignGAME
from qdots_qll.models.single_dot_weak_coupling_GAME import (
    SingleDotWeakCouplingGAME,
)
from data import Data
from experiments import ExperimentSingleDotWeakCouplingGAME
from qdots_qll.resamplers import LiuWestResampler, MetropolisSampler
from qdots_qll.smc import SMCUpdater, replace_single_datum, replace_single_exp

model = SingleDotWeakCouplingGAME()


DEBUG:2024-09-03 16:41:04,457:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:41:04,458:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:41:04,463:jax._src.lru_cache:107: Cache hit for key: 'jit_convert_element_type-0581db0bf206ecfadd3967c7aa8a468505e540523fe3dc2b0e3e9ca72cd76bca'
DEBUG:2024-09-03 16:41:04,490:jax._src.compiler:98: Persistent compilation cache hit for 'jit_convert_element_type' with key 'jit_convert_element_type-0581db0bf206ecfadd3967c7aa8a468505e540523fe3dc2b0e3e9ca72cd76bca'
DEBUG:2024-09-03 16:41:04,610:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:41:04,611:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:41:04,612:jax._src.lru_cache:107: Cache hit fo

In [2]:
key = jax.random.key(0)
key, subkey = jax.random.split(key)

boundaries = jnp.array(
    [
        [0.1, 0.5],
        [0.1, 0.5],
        [0.01, 0.2],
        [-0.5, -0.01],
    ]
)

loc = boundaries.mean(axis=1)
scale = boundaries.std(axis=1)

truncated_norm = tfp.distributions.TruncatedNormal(
    loc=loc, scale=scale / 1.5, low=boundaries[:, 0], high=boundaries[:, 1]
)

no_particles = 500
init_particles_locations = truncated_norm.sample(
    seed=subkey, sample_shape=(no_particles,)
)
weights = jnp.ones(no_particles) / no_particles

dist = Distribution(particles_locations=init_particles_locations, weights=weights)

e1 = ExperimentSingleDotWeakCouplingGAME(13.3, 0, 0)
key, subkey = jax.random.split(key)
with jax.log_compiles():
    re = model.measure_one_experiment(subkey, e1)
    re.block_until_ready()

e2 = ExperimentSingleDotWeakCouplingGAME(13.0, 0, 0)
key, subkey = jax.random.split(key)
with jax.log_compiles():
    re = model.measure_one_experiment(subkey, e2)
    re.block_until_ready()

    d2 = Data(e2, re)
    model.log_lkl_single_datum(model.true_parameters, d2)
N_max_exps = 10000
ExperimentSingleDotWeakCouplingGAME(0.0, 0, 0)


DEBUG:2024-09-03 16:41:08,159:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:41:08,160:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:41:08,161:jax._src.lru_cache:107: Cache hit for key: 'jit_convert_element_type-28715c8e398af27a27cbadfd85639e455d88b883589491c29fe53cf23f5f8587'
DEBUG:2024-09-03 16:41:08,163:jax._src.compiler:98: Persistent compilation cache hit for 'jit_convert_element_type' with key 'jit_convert_element_type-28715c8e398af27a27cbadfd85639e455d88b883589491c29fe53cf23f5f8587'
DEBUG:2024-09-03 16:41:08,170:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:41:08,170:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:41:08,172:jax._src.lru_cache:107: Cache hit fo

ExperimentSingleDotWeakCouplingGAME(
  time=f32[1],
  initial_state=i8[1],
  measurement_basis=i8[1]
)

In [3]:
@jax.jit
def f_to_try_compilation(full_data):
    a = full_data[5]

    return jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
        jnp.repeat(jnp.array([-9999, -8888, -9999])[None, :], N_max_exps, axis=0)
    )


empty_experiments = jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
    jnp.repeat(jnp.array([-9999, -9999, -9999])[None, :], N_max_exps, axis=0)
)

empty_outcomes = jnp.ones(N_max_exps) * -9999

empty_data = jax.vmap(lambda exp, outcome: Data(exp, outcome), in_axes=(0, 0))(
    empty_experiments, empty_outcomes
)


@jax.jit
def f_to_try_compilation_2(iteration, full_data, data_to_replace):
    return replace_single_datum(full_data, data_to_replace, iteration)


DEBUG:2024-09-03 16:41:31,721:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:41:31,722:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:41:31,723:jax._src.lru_cache:107: Cache hit for key: 'jit_convert_element_type-56bd62759222df2b7cddc9c0ebe2dda64fa49b86a820a26d82dae59c675f0170'
DEBUG:2024-09-03 16:41:31,725:jax._src.compiler:98: Persistent compilation cache hit for 'jit_convert_element_type' with key 'jit_convert_element_type-56bd62759222df2b7cddc9c0ebe2dda64fa49b86a820a26d82dae59c675f0170'
DEBUG:2024-09-03 16:41:31,746:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:41:31,747:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:41:31,748:jax._src.lru_cache:107: Cache hit fo

In [4]:

empty_experiments = jax.vmap(lambda e: ExperimentSingleDotWeakCouplingGAME(*e))(
    jnp.repeat(jnp.array([13.0, 0, 0])[None, :], N_max_exps, axis=0)
)

empty_outcomes = jnp.ones(N_max_exps) * 0

empty_data = jax.vmap(lambda exp, outcome: Data(exp, outcome), in_axes=(0, 0))(
    empty_experiments, empty_outcomes
)


DEBUG:2024-09-03 16:42:30,277:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:42:30,279:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:42:30,280:jax._src.lru_cache:104: Cache miss for key: 'jit_convert_element_type-71f4c93c25018fadca64f6aea860d626cdbd72f5c32560ed2730d558a33c2250'
DEBUG:2024-09-03 16:42:30,406:jax._src.compiler:704: 'jit_convert_element_type' took at least 0.00 seconds to compile (0.12s)
DEBUG:2024-09-03 16:42:30,415:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-03 16:42:30,416:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-03 16:42:30,417:jax._src.lru_cache:104: Cache miss for key: 'jit_broadcast_in_dim-3117aecee40cc7c97555c8d997fe5ece592d9dfc88196ceeb8f06d24fe8d

In [9]:
empty_data[390].experiment.time

Array([13.], dtype=float32)

In [6]:
from qdots_qll.distributions import update_log_weights


@jax.jit
def f2(key, model, iteration, all_data, distribution, *args, **kwargs):
    key, subkey = jax.random.split(key)
    # experiment = e1
    experiment = exp_design.generate_experiment(
        model=model,
        distribution=distribution,
        data=all_data,
        subkey=subkey,
    )

    outcome = model.measure_one_experiment(subkey, experiment)
    datum = Data(experiment, outcome)

    log_lkl = model.log_lkl_datum_multiple_particles(
        distribution.particles_locations, datum
    )

    all_data = replace_single_datum(all_data, datum, iteration)

    new_dist: Distribution = update_log_weights(dist=dist, new_log_lkl=log_lkl)
    iteration = iteration + 1
    return key, model, iteration, all_data, new_dist

In [7]:
resampler = LiuWestResampler(boundaries)

In [8]:
@jax.jit
def f_to_see_slicing(iteration, all_data):
    zeros = jnp.zeros(len(all_data))
    aux = jnp.arange(len(all_data))
    this_thing = jnp.where(aux < iteration, all_data.outcome, zeros)
    return this_thing


a = jnp.ones(1000) * 0
b = jnp.ones(1000) * 3

aux = jnp.arange(len(a))

for i in range(100):
    empty_data = replace_single_datum(empty_data, d2, i)

f_to_see_slicing(100, empty_data)

model.log_lkl_single_datum(model.true_parameters, empty_data[0])

In [17]:
jax.jit(model.batch_total_log_lkl)(jnp.array(10, int), dist.particles_locations, data=empty_data)

TypeError: unhashable type: 'jaxlib.xla_extension.ArrayImpl'

In [13]:
# xs is all the data
# length is gonna be the i index


@jax.jit
def f_to_scan(carry, x, data, ):
    datum = data[carry]
    loglkl = model.log_lkl_single_datum(model.true_parameters, datum)
    carry = carry + 1
    return carry, loglkl


In [14]:
model.log_lkl_single_datum(model.true_parameters, empty_data[5])

Array([-0.6948499], dtype=float32)

In [29]:
jax.lax.scan(
    lambda carry, x: f_to_scan(carry, x, empty_data),
    init=jnp.array([0]),
    length=100
    # xs=jnp.arange(2500),
)

DEBUG:2024-09-02 20:07:19,606:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-02 20:07:19,607:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-02 20:07:19,609:jax._src.lru_cache:107: Cache hit for key: 'jit_scan-4c419941a91598314a350f6b994f9dde749c3e21d07c39a93ea661c3e080d1e7'
DEBUG:2024-09-02 20:07:19,634:jax._src.compiler:98: Persistent compilation cache hit for 'jit_scan' with key 'jit_scan-4c419941a91598314a350f6b994f9dde749c3e21d07c39a93ea661c3e080d1e7'


(Array([100], dtype=int32),
 Array([[[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[-0.6948499]],
 
        [[

In [ ]:
model

In [42]:
jax.lax.map(
    lambda datum: model.log_lkl_datum_multiple_particles(dist.particles_locations, datum),
    empty_data,
    batch_size=2000,
)

DEBUG:2024-09-02 20:12:49,638:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-02 20:12:49,639:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-02 20:12:49,642:jax._src.lru_cache:107: Cache hit for key: 'jit_scan-535d7b46441cac04e4d6660f50302d9dc002ad6ba60b3d6213f2708d067ec3ad'
DEBUG:2024-09-02 20:12:49,679:jax._src.compiler:98: Persistent compilation cache hit for 'jit_scan' with key 'jit_scan-535d7b46441cac04e4d6660f50302d9dc002ad6ba60b3d6213f2708d067ec3ad'


Array([[-0.9922905, -0.5402967, -0.6685371, ..., -0.5182849, -0.4524886,
        -0.8495726],
       [-0.9922905, -0.5402967, -0.6685371, ..., -0.5182849, -0.4524886,
        -0.8495726],
       [-0.9922905, -0.5402967, -0.6685371, ..., -0.5182849, -0.4524886,
        -0.8495726],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan]], dtype=float32)

In [106]:
# using map

@jax.jit
def f_to_scan(carry, x, data, particles):
    datum = data[carry]

    loglkl = model.log_lkl_datum_multiple_particles(particles, datum)
    carry = carry + 1
    return carry, loglkl


a, b = jax.lax.scan(
    lambda carry, x: f_to_scan(carry, x, empty_data, dist.particles_locations),
    init=jnp.array([0]),
    length=10
    # xs=jnp.arange(2500),
)



DEBUG:2024-09-02 20:33:41,818:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-02 20:33:41,819:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-02 20:33:41,822:jax._src.lru_cache:107: Cache hit for key: 'jit_scan-5f221d0a300398451eaf18621fe6c9f9f5b4a0b7a7dc40d3bd9f53a2f89712d6'
DEBUG:2024-09-02 20:33:41,860:jax._src.compiler:98: Persistent compilation cache hit for 'jit_scan' with key 'jit_scan-5f221d0a300398451eaf18621fe6c9f9f5b4a0b7a7dc40d3bd9f53a2f89712d6'


In [111]:
b.sum(axis=0).shape

(500,)

In [107]:
a

Array([10], dtype=int32)

# Optimizing resamplers

In [71]:
from qdots_qll.resamplers import Resampler

from jaxtyping import Array, Complex, Float, Int, Real



In [13]:
resampler = MetropolisSampler(boundaries, model)

In [14]:
jnp.array(7)

Array(7, dtype=int32, weak_type=True)

In [17]:
model.batch_total_log_lkl(10, dist.particles_locations, empty_data)

TypeError: Indexer must have integer or boolean type, got indexer with type float32 at position 0, indexer value Traced<ShapedArray(float32[])>with<DynamicJaxprTrace(level=1/0)>

In [15]:
key, subkey = jax.random.split(key)
resampler.resample(subkey, jnp.array(10, int), dist, empty_data)

ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape int32[]
The problem arose with the `int` function. If trying to convert the data type of a value, try using `x.astype(int)` or `jnp.array(x, int)` instead.
The error occurred while tracing the function resample at /home/antonio/dev/qdots_efficient/qdots_qll/resamplers.py:354 for jit. This concrete value was not available in Python because it depends on the value of the argument index_data.

See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.ConcretizationTypeError

In [72]:
class myMetropolisSampler(Resampler):
    factor: float
    boundaries: Array
    model: eqx.Module

    def __init__(self, boundaries: Array, model: eqx.Module, factor=1):
        self.factor = factor
        self.boundaries = boundaries
        self.model = model

In [73]:
@jax.jit
def _multinomial_importance_sampling(subkey, dist):
    no_particles = dist.particles_locations.shape[0]

    new_locs = jax.random.choice(
        subkey,
        dist.particles_locations,
        shape=(no_particles,),
        p=dist.weights / dist.weights.sum(),
    )

    return new_locs


In [103]:
key, subkey = jax.random.split(key)

_multinomial_importance_sampling(subkey, dist)

Array([[ 0.47101402,  0.10185759,  0.08841481, -0.03066938],
       [ 0.12267141,  0.13371082,  0.03691489, -0.18282732],
       [ 0.24109134,  0.41551265,  0.10103706, -0.2765727 ],
       ...,
       [ 0.10389079,  0.29846966,  0.13241813, -0.38600308],
       [ 0.18116932,  0.24340752,  0.05968317, -0.456763  ],
       [ 0.1370224 ,  0.27329296,  0.08709144, -0.48138416]],      dtype=float32)

In [63]:
len(empty_data)

10000

In [68]:
jax.lax.map(
    lambda datum: model.log_lkl_datum_multiple_particles(
        dist.particles_locations, datum
    ),
    xs=empty_data,
    batch_size=4000,
)

DEBUG:2024-09-02 20:27:07,506:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-02 20:27:07,507:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-02 20:27:07,509:jax._src.lru_cache:104: Cache miss for key: 'jit_dynamic_slice-03a7ce2cf0721a0a1fb3e676568530362ff4812efadfa9a7fbd2898345cca704'
DEBUG:2024-09-02 20:27:07,532:jax._src.compiler:704: 'jit_dynamic_slice' took at least 0.00 seconds to compile (0.02s)
DEBUG:2024-09-02 20:27:07,543:jax._src.compiler:166: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[CudaDevice(id=0)]]
DEBUG:2024-09-02 20:27:07,543:jax._src.compiler:230: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-02 20:27:07,545:jax._src.lru_cache:104: Cache miss for key: 'jit_reshape-d824079bd687ae18a77b6fd3d726a57d71ab6f757f98c0e86c9e0d4ac11cc79f'
DEBUG:2024-09-02 

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 2326004776 bytes.

In [26]:
jax.__version__

'0.4.28'

In [78]:
empty_data[5]

Data(
  experiment=ExperimentSingleDotWeakCouplingGAME(
    time=f32[1],
    initial_state=i8[1],
    measurement_basis=i8[1]
  ),
  outcome=i8[1]
)

In [85]:
f_to_scan(0.0, 90, empty_data)

(Array([-0.6948499], dtype=float32), Array([-0.6948499], dtype=float32))

In [27]:
empty_data[140].outcome

Array([-128], dtype=int8)

In [60]:
d2.experiment.time

Array([13.], dtype=float32)

DEBUG:2024-09-02 19:50:16,394:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-09-02 19:50:16,395:jax._src.compiler:203: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-09-02 19:50:16,400:jax._src.compiler:287: Persistent compilation cache hit for 'jit_replace_single_datum'


In [66]:
empty_data.outcome[2]

Array([0], dtype=int8)

In [38]:
empty_data[0]

Data(
  experiment=ExperimentSingleDotWeakCouplingGAME(
    time=f32[1],
    initial_state=i8[1],
    measurement_basis=i8[1]
  ),
  outcome=i8[1]
)

In [43]:
a, b = f_to_scan(jnp.array([0]), 0)

In [47]:
a

Array([nan], dtype=float32)

In [ ]:
model.batch_total_log_lkl

In [180]:
empty_data.outcome.at[0:10].set(999)

DEBUG:2024-08-30 17:51:11,243:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-08-30 17:51:11,244:jax._src.compiler:203: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-08-30 17:51:11,263:jax._src.compiler:557: 'jit_broadcast_in_dim' took at least 0.00 seconds to compile (0.02s)
DEBUG:2024-08-30 17:51:11,268:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-08-30 17:51:11,269:jax._src.compiler:203: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-08-30 17:51:11,275:jax._src.compiler:557: 'jit__squeeze' took at least 0.00 seconds to compile (0.01s)
DEBUG:2024-08-30 17:51:11,282:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-08-30 17:51:11,283:jax._src.compiler:203: get_compile_options XLA-AutoFDO 

Array([[ -25],
       [ -25],
       [ -25],
       ...,
       [-128],
       [-128],
       [-128]], dtype=int8)

In [ ]:
jnp.where()

In [174]:
len(empty_data)

10000

In [173]:
f_to_see_slicing(200, empty_data)

DEBUG:2024-08-30 17:48:42,296:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-08-30 17:48:42,297:jax._src.compiler:203: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-08-30 17:48:42,320:jax._src.compiler:557: 'jit_dynamic_slice' took at least 0.00 seconds to compile (0.02s)
DEBUG:2024-08-30 17:48:42,329:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-08-30 17:48:42,330:jax._src.compiler:203: get_compile_options XLA-AutoFDO profile: using XLA-AutoFDO profile version -1
DEBUG:2024-08-30 17:48:42,362:jax._src.compiler:557: 'jit_dynamic_slice' took at least 0.00 seconds to compile (0.03s)
DEBUG:2024-08-30 17:48:42,378:jax._src.compiler:144: get_compile_options: num_replicas=1 num_partitions=1 device_assignment=[[cuda(id=0)]]
DEBUG:2024-08-30 17:48:42,382:jax._src.compiler:203: get_compile_options XLA-AutoFD

Data(
  experiment=ExperimentSingleDotWeakCouplingGAME(
    time=f32[10000,1],
    initial_state=i8[10000,1],
    measurement_basis=i8[10000,1]
  ),
  outcome=i8[10000,1]
)

In [160]:
for i in range(1000):
    key, subkey = jax.random.split(key)

    dist = resampler.resample(subkey, dist)
    print(dist.ev())

[ 0.34508193  0.28735885  0.12490679 -0.2946432 ]
[ 0.34558985  0.286718    0.12516737 -0.29839423]
[ 0.35010445  0.28502107  0.12629618 -0.30005386]
[ 0.34946632  0.28320503  0.12744579 -0.30154288]
[ 0.35272688  0.28259957  0.12820226 -0.3029058 ]
[ 0.3545184   0.28482223  0.1281274  -0.30326116]
[ 0.35566777  0.2914218   0.12735295 -0.3041926 ]
[ 0.35870016  0.29457223  0.12680134 -0.30119362]
[ 0.35873908  0.29256186  0.1264385  -0.30237076]
[ 0.36099714  0.29285192  0.12633963 -0.30554086]
[ 0.3601289   0.29422283  0.12642096 -0.3048944 ]
[ 0.3596338   0.2943917   0.12612489 -0.30535722]
[ 0.36041903  0.29550904  0.12570485 -0.3091253 ]
[ 0.36010167  0.29623014  0.12519166 -0.30581385]
[ 0.358464    0.29398042  0.12427747 -0.30556738]
[ 0.35709554  0.2975484   0.12447709 -0.30467317]
[ 0.35698414  0.2924909   0.12422539 -0.30215558]
[ 0.3588966   0.28977072  0.12373829 -0.30316952]
[ 0.35849267  0.29080838  0.12442902 -0.3040725 ]
[ 0.35759264  0.29303792  0.12492274 -0.29887208]


In [154]:
dist.particles_locations

Array([[ 0.31357154,  0.23828031,  0.0432434 , -0.3119301 ],
       [ 0.32178417,  0.2632918 ,  0.07995646, -0.37498304],
       [ 0.38519037,  0.27206308,  0.06930517, -0.1123676 ],
       ...,
       [ 0.30942082,  0.23245247,  0.11418836, -0.32484484],
       [ 0.33002204,  0.27239776,  0.10863972, -0.23955849],
       [ 0.32685995,  0.25807038,  0.08140806, -0.34576234]],      dtype=float32)

In [143]:
dist.no_particles

500

In [120]:
exp_design = RandExpDesignGAME()

In [124]:
iteration = 0
for _ in range(10):
    key, model, iteration, all_data, new_dist = f2(
        key, model, iteration, empty_data, dist
    )

In [68]:
for iteration in range(1000):
    empty_data = f_to_try_compilation_2(iteration, empty_data, d2)

In [69]:
empty_data.outcome

Array([[   0],
       [   0],
       [   0],
       ...,
       [-128],
       [-128],
       [-128]], dtype=int8)

In [98]:
dist.particles_locations

Array([[ 0.18209201,  0.46332115,  0.1516649 , -0.3411265 ],
       [ 0.33036196,  0.25449303,  0.18003954, -0.27368462],
       [ 0.36283004,  0.35961238,  0.0131733 , -0.20299599],
       ...,
       [ 0.3230243 ,  0.19129181,  0.19682081, -0.10106033],
       [ 0.40749988,  0.17357248,  0.14290242, -0.10408178],
       [ 0.108294  ,  0.2157959 ,  0.03785142, -0.04795896]],      dtype=float32)

In [18]:
a, b = jax.tree.flatten(empty_data)

In [52]:
@jax.jit
def replace_single_exp(all_exps, exp_to_be_placed, position):
    arrs, structure = jax.tree.flatten(all_exps)
    arr_e, _ = jax.tree.flatten(exp_to_be_placed)
    for j in range(len(arrs)):
        arrs[j] = arrs[j].at[position].set(arr_e[j])
    new_empty = jax.tree.unflatten(structure, arrs)
    return new_empty


@jax.jit
def replace_single_datum(all_data, datum_to_be_placed, position):
    structure_all_data = eqx.tree_flatten_one_level(all_data)[1]

    all_exps = all_data.experiment
    all_outcomes = all_data.outcome

    exp_to_be_placed = datum_to_be_placed.experiment
    outcome_to_be_placed = datum_to_be_placed.outcome

    new_outcomes = all_outcomes.at[position].set(outcome_to_be_placed)

    replaced_exps = replace_single_exp(all_exps, exp_to_be_placed, position)

    return jax.tree.unflatten(structure_all_data, [replaced_exps, new_outcomes])
